In [138]:
# ruff: noqa: F401, F403

import dataclasses
import os
import subprocess
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch

from IPython.display import *
from plotly.subplots import make_subplots

from pacer import (
    CoordinateSystem,
    # read_dat_file,
    DatVersion,
    GPMFSource,
    GPSSample,
    Lap,
    Laps,
    Point,
    PointInTime_GPSSample,
    RawGPSSource,
    Segment,
    SequentialGPSSource,
    Vec3f,
)

In [24]:
file_paths = [
    "/Users/denys/dev/pacer/videos/clay-pigeon/GX010307.MP4",
    "/Users/denys/dev/pacer/videos/clay-pigeon/GX020307.MP4",
]

In [64]:
def correct_timestamps(
    samples: list[GPSSample], cs: CoordinateSystem
) -> list[GPSSample]:
    attributes: list[str] = [
        attr for attr in samples[0].__dir__() if not attr.startswith("_")
    ]
    df = pd.DataFrame(
        [
            {k: getattr(s, k) for k in attributes}
            | {p: getattr(cs.local(s), p) for p in "xyz"}
            for s in samples
        ]
    )
    df = df.assign(timestamp=lambda d: pd.to_datetime(d["timestamp_ms"], unit="ms"))
    df = df.assign(
        dist_prev=lambda d: (
            d["x"].diff().pow(2) + d["y"].diff().pow(2) + d["z"].diff().pow(2)
        ).pow(0.5),
        time_prev=lambda d: d["dist_prev"] / d["full_speed"],
    )

    dataset = df.iloc[200:-200].loc[
        lambda d: (
            (d["timestamp"] == d["timestamp"].shift(+1))
            | (d["timestamp"] == d["timestamp"].shift(-1))
        )
    ]
    dataset.pipe(lambda d: px.scatter(d, y="time_prev"))

    dataset = dataset.assign(
        di=lambda d: (d["time_prev"] / d["time_prev"].median()).round()
    )
    total_points = dataset["di"].sum()
    total_time = dataset["timestamp"].max() - dataset["timestamp"].min()
    single_step = total_time / (total_points - 9)
    dataset = dataset.assign(
        timestamp_corrected=lambda d: (
            dataset["timestamp"].iloc[0] + (d["di"] * single_step).cumsum()
        )
    )

    df = df.assign(
        timestamp_corrected=dataset["timestamp_corrected"]
        .reindex(df.index)
        .fillna(df["timestamp"])
    )

    return [
        GPSSample(
            lat=row["lat"],
            lon=row["lon"],
            altitude=row["altitude"],
            full_speed=row["full_speed"],
            ground_speed=row["ground_speed"],
            timestamp_ms=row["timestamp_corrected"].as_unit("ns").value // 10**6,
        )
        for _, row in df.iterrows()
    ]

In [ ]:
@dataclasses.dataclass
class Session:
    start_time: pd.Timestamp
    single_files: dict[str, GPMFSource]
    starts_of_files: dict[str, pd.Timestamp]
    ends_of_files: dict[str, pd.Timestamp]

    laps: Laps
    cs: CoordinateSystem

    def locate_timestamp(
        self, timestamp: pd.Timestamp | float
    ) -> tuple[str, pd.Timedelta]:
        if isinstance(timestamp, float):
            timestamp = self.start_time + pd.to_timedelta(timestamp, unit="s")
        for f in self.single_files:
            if self.start_of_file[f] <= timestamp < self.end_of_file[f]:
                return f, timestamp - self.start_of_file[f]

    @classmethod
    def from_files(
        cls, file_paths: list[str], cs: CoordinateSystem | None = None
    ) -> "Session":
        single_files = {f: GPMFSource(f) for f in file_paths}

        samples = []
        start_of_file = {}
        end_of_file = {}

        for fn, f in single_files.items():
            total_duration = f.get_total_duration()

            def on_sample(s, _, _2):
                if fn not in start_of_file:
                    start_of_file[fn] = pd.Timestamp(s.timestamp_ms, unit="ms")
                end_of_file[fn] = pd.Timestamp(s.timestamp_ms, unit="ms")
                # if s.full_speed > 3:
                samples.append(s)

            while not f.is_end():
                f.read_samples(on_sample)
                f.next()

        if cs is None:
            mean_point = GPSSample(
                lat=np.mean([s.lat for s in samples]),
                lon=np.mean([s.lon for s in samples]),
                altitude=np.mean([s.altitude for s in samples]),
            )

            cs = CoordinateSystem(mean_point)

        samples = correct_timestamps(samples, cs)
        laps = Laps()
        start_time = pd.to_datetime(samples[0].timestamp_ms, unit="ms")
        for s in samples:
            laps.add_point(
                s,
                (
                    pd.to_datetime(s.timestamp_ms, unit="ms") - start_time
                ).total_seconds(),
            )
        laps.set_coordinate_system(cs)

        return cls(
            start_time=start_time,
            single_files=single_files,
            starts_of_files=start_of_file,
            ends_of_files=end_of_file,
            laps=laps,
            cs=cs,
        )

    def set_start_plot_map(self, start_line: Segment, *sectors: Segment) -> None:
        fig = px.line(
            x=[
                self.cs.local(s).x
                for i in range(self.laps.point_count())
                if (s := self.laps.get_point(i).point) is not None
            ],
            y=[
                self.cs.local(s).y
                for i in range(self.laps.point_count())
                if (s := self.laps.get_point(i).point) is not None
            ],
        )

        self.laps.sectors.start_line = start_line
        self.laps.sectors.sector_lines = sectors
        self.laps.update()

        for line in [self.laps.sectors.start_line] + self.laps.sectors.sector_lines:
            fig.add_trace(
                px.line(
                    x=[line.first.x, line.second.x], y=[line.first.y, line.second.y]
                ).data[0]
            )

        fig.update_layout(title=f"Laps: {self.laps.laps_count()}")

        return fig

    def get_laptimes(self, threshold: float = 0.25) -> pd.DataFrame:
        fraction = 1 + threshold

        n = self.laps.laps_count()
        s = self.laps.sector_count() + 1

        laptimes = np.array([self.laps.lap_time(i) for i in range(n)]).reshape((n, 1))
        sectors = np.array([self.laps.sector_time(i) for i in range(s * n)]).reshape(
            (n, s)
        )

        columns = ["laptime"] + [f"S{i}" for i in range(1, s + 1)]
        df = pd.DataFrame(np.concat([laptimes, sectors], axis=1), columns=columns)

        return df.where(lambda d: d > 0.01).where(
            lambda d: d < fraction * d.min(axis=0)
        )

    def plot_laptimes(self, threshold: float = 0.25) -> None:
        laptimes = self.get_laptimes(threshold)
        return px.line(
            laptimes["laptime"],
            markers=True,
            title=f"Laptimes (limited to {int(100 * (1 + threshold))}% of the best ({np.nanmin(laptimes):.3f}))",
        )

In [223]:
session = Session.from_files(file_paths, None)

In [243]:
session.set_start_plot_map(
    Segment(first=Point(x=15, y=6), second=Point(x=30, y=-10)),
    Segment(first=Point(x=-55, y=-6), second=Point(x=-45, y=10)),
    Segment(first=Point(x=-44, y=11), second=Point(x=-32, y=17)),
    Segment(first=Point(x=4, y=110), second=Point(x=12, y=88)),
).update_layout(height=1000)

In [244]:
laptimes = session.get_laptimes()

In [245]:
px.line(laptimes, markers=True)

In [247]:
lap_30, lap_37 = session.laps.get_lap(30), session.laps.get_lap(37)

In [70]:
len(lap_30.points), len(lap_37.points)

(780, 777)

In [249]:
lap_30.width = 10
# lap_37.width = 10
lap_37_30 = lap_30.resample(lap_37, session.cs)
lap_23_30 = lap_30.resample(session.laps.get_lap(23), session.cs)
lap_26_30 = lap_30.resample(session.laps.get_lap(26), session.cs)
len(lap_30.points), len(lap_37.points), len(lap_37_30.points), len(lap_23_30.points), len(lap_26_30.points)

(780, 777, 780, 780, 780)

In [266]:
def toframe(lap: Lap) -> pd.DataFrame:
    def todict(
        s: GPSSample, t: float, d: float, cs: CoordinateSystem
    ) -> dict[str, float]:
        p = cs.local(s)

        return (
            {
                a: getattr(s, a)
                for a in ["lat", "lon", "altitude", "full_speed", "ground_speed"]
            }
            | {a: getattr(p, a) for a in "xyz"}
            | {"timestamp": t, "dist": d}
        )

    return pd.DataFrame(
        [
            todict(pt.point, pt.time, dist, session.cs)
            for pt, dist in zip(lap.points, lap.cum_distances)
        ]
    ).set_index("dist")


f_30 = toframe(lap_30)
f_37 = toframe(lap_37_30)
f_23 = toframe(lap_23_30)
f_26 = toframe(lap_26_30)

frames = {
    30: f_30,
    37: f_37,
    # 23: f_23,
    # 26: f_26,
}

In [267]:
delta = pd.concat(
    [
        (f["timestamp"] - f_37["timestamp"])
        .pipe(lambda s: s - s.iloc[0])
        .rename(f"delta {i} - 37")
        .to_frame()
        for i, f in frames.items()
    ],
    axis=1,
)


fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.05,
    subplot_titles=(
        f"Speed comparison {', '.join(f'lap {i}: {session.laps.lap_time(i):.3f}' for i in frames)}",
        "Delta comparison",
    ),
)

for trace in px.line(
    pd.concat(
        [f["full_speed"].rename(f"speed_{i}").to_frame() for i, f in frames.items()]
    )
).data:
    fig.add_trace(trace, row=1, col=1)
for trace in px.line(delta).data:
    fig.add_trace(trace, row=2, col=1)

points = pd.concat(
    [delta.diff().rolling(20).mean().clip(-0.001, +0.001)]
    + [
        f[["x", "y"]].rename(columns={"x": f"x_{i}", "y": f"y_{i}"})
        for i, f in frames.items()
    ],
    axis=1,
)

for i in frames:
    fig.add_trace(
        px.scatter(
            points,
            x=f"x_{i}",
            y=f"y_{i}",
            color=f"delta {i} - 37",
            hover_data={
                "dist": frames[i].index,
                "lap": np.ones(len(points)) * i,
                **{f"speed_{i}": frames[i]["full_speed"] for i in frames},
            },
            title=f"lap {i}",
        ).data[0],
        row=3,
        col=1,
    )

fig.update_layout(height=1000)

# Didn't work :(

In [ ]:
file_paths = ["/Users/denys/dev/pacer/videos/clay-pigeon/GX010130.MP4"]